# Lesson 2: Langchain Expression language

In [ ]:
import os
from dotenv import load_dotenv

load_dotenv()

import warnings
warnings.filterwarnings("ignore", category=DeprecationWarning)

# Default model is Haiku 4.5; override with e.g. LLM_MODEL=claude-opus-4-8
MODEL = os.environ.get("LLM_MODEL", "claude-haiku-4-5")

# Simple Chain

In [ ]:
from langchain.prompts import ChatPromptTemplate
from langchain_anthropic import ChatAnthropic
from langchain.schema.output_parser import StrOutputParser

prompt = ChatPromptTemplate.from_template("tell me a joke about {topic}")
model = ChatAnthropic(model=MODEL)
output_parser = StrOutputParser()

chain = prompt | model | output_parser
chain.invoke({"topic": "bears"})

# More complex chain

In [ ]:
# Anthropic has no embeddings API, so this RAG example uses a local
# (provider-agnostic) HuggingFace embedding model instead of OpenAI's.
from langchain_huggingface import HuggingFaceEmbeddings
from langchain_community.vectorstores import DocArrayInMemorySearch

vectorstore = DocArrayInMemorySearch.from_texts(
    ["harrison worked at kensho", "bears like to eat honey"],
    embedding=HuggingFaceEmbeddings(model_name="sentence-transformers/all-MiniLM-L6-v2")
)
retriever = vectorstore.as_retriever()

retriever.invoke("where did harrison work?")

In [5]:
retriever.invoke("what do bears like to eat?")

[Document(metadata={}, page_content='bears like to eat honey'),
 Document(metadata={}, page_content='harrison worked at kensho')]

In [6]:
template = """Answer the question based only on the following context:
{context}

Question:{question}
"""

prompt = ChatPromptTemplate.from_template(template)

In [8]:
from langchain.schema.runnable import RunnableMap

chain = RunnableMap({
    "context": lambda x: retriever.get_relevant_documents(x["question"]),
    "question": lambda x: x["question"]
}) | prompt | model | output_parser

chain.invoke({"question": "where did harrison work?"})

'Harrison worked at Kensho.'

In [9]:
inputs = RunnableMap({
    "context": lambda x: retriever.get_relevant_documents(x["question"]),
    "question": lambda x: x["question"]
})
inputs.invoke({"question": "where did harrison work?"})

{'context': [Document(metadata={}, page_content='harrison worked at kensho'),
  Document(metadata={}, page_content='bears like to eat honey')],
 'question': 'where did harrison work?'}

# Bind

In [10]:
functions = [
    {
        "name": "weather_search",
        "description": "Search for weather given an airport code",
        "parameters": {
            "type": "object",
            "properties": {
                "airport_code": {
                    "type": "string",
                    "description": "The airport code to get the weather for"
                },
            },
            "required": ["airport_code"]
        }
    }
]

In [ ]:
prompt = ChatPromptTemplate.from_messages(
    [
        ("human", "{input}")
    ]
)

# Provider-agnostic tool binding via .bind_tools (replaces OpenAI's .bind(functions=...))
model = ChatAnthropic(model=MODEL, temperature=0).bind_tools(functions)

In [12]:
runnable = prompt | model

runnable.invoke({"input": "what is the weather in sf"})

AIMessage(content='', additional_kwargs={'function_call': {'arguments': '{"airport_code":"SFO"}', 'name': 'weather_search'}, 'refusal': None}, response_metadata={'token_usage': {'completion_tokens': 16, 'prompt_tokens': 64, 'total_tokens': 80, 'completion_tokens_details': {'accepted_prediction_tokens': 0, 'audio_tokens': 0, 'reasoning_tokens': 0, 'rejected_prediction_tokens': 0}, 'prompt_tokens_details': {'audio_tokens': 0, 'cached_tokens': 0}}, 'model_name': 'gpt-3.5-turbo-0125', 'system_fingerprint': None, 'id': 'chatcmpl-C2iPTpjG7AHWcKBsscswQdOS6PIeo', 'service_tier': 'default', 'finish_reason': 'function_call', 'logprobs': None}, id='run--8ad9a5fd-8876-49d2-81c6-2cfde004c5d2-0', usage_metadata={'input_tokens': 64, 'output_tokens': 16, 'total_tokens': 80, 'input_token_details': {'audio': 0, 'cache_read': 0}, 'output_token_details': {'audio': 0, 'reasoning': 0}})

In [13]:
functions = [
    {
        "name": "weather_search",
        "description": "Search for weather given an airport code",
        "parameters": {
            "type": "object",
            "properties": {
                "airport_code": {
                    "type": "string",
                    "description": "The airport code to get the weather for"
                },
            },
            "required": ["airport_code"]
        }
    },
    {
        "name": "sports_search",
        "description": "Search for news of recent sport events",
        "parameters": {
            "type": "object",
            "properties": {
                "team_name": {
                    "type": "string",
                    "description": "The sports team to search for"
                },
            },
            "required": ["team_name"]
        }
    }
]

In [ ]:
model = ChatAnthropic(model=MODEL, temperature=0).bind_tools(functions)

runnable = prompt | model
runnable.invoke({"input": "how did the patriots do yesterday?"})

# Fallbacks

In [ ]:
from langchain_anthropic import ChatAnthropic
from langchain.schema.output_parser import StrOutputParser
import json

simple_model = ChatAnthropic(
    model=MODEL,
    temperature=0,
    max_tokens=1000,
)

In [ ]:
def ensure_valid_json(response):
    try:
        parsed_response = json.loads(response)
        return parsed_response
    except json.JSONDecodeError:
        return {"error": "Failed to decode JSON"}


simple_chain = simple_model | StrOutputParser() | ensure_valid_json

challenge = """write three poems in a JSON blob, where each poem is a JSON blob with keys 'title', 'author', and 'first_line'.
Output the entire response in a valid JSON format.
"""

simple_chain.invoke(challenge)

In [ ]:
model = ChatAnthropic(model=MODEL, temperature=0)
chain = model | StrOutputParser() | json.loads

In [18]:
chain.invoke(challenge)

{'poems': [{'title': 'The Rose',
   'author': 'Emily Dickinson',
   'first_line': 'A rose is a rose is a rose'},
  {'title': 'The Road Not Taken',
   'author': 'Robert Frost',
   'first_line': 'Two roads diverged in a yellow wood'},
  {'title': 'Hope is the Thing with Feathers',
   'author': 'Emily Dickinson',
   'first_line': 'Hope is the thing with feathers'}]}

In [19]:
final_chain = simple_chain.with_fallbacks([chain])

In [20]:
final_chain.invoke(challenge)

{'poem1': {'title': 'Autumn Leaves',
  'author': 'Jane Smith',
  'first_line': 'Crisp leaves crunch underfoot'},
 'poem2': {'title': "The Ocean's Song",
  'author': 'John Doe',
  'first_line': 'Waves crash against the shore'},
 'poem3': {'title': "A Winter's Night",
  'author': 'Emily Jones',
  'first_line': 'Snowflakes dance in the moonlight'}}

# Interface

In [ ]:
prompt = ChatPromptTemplate.from_template("tell me a joke about {topic}")
model = ChatAnthropic(model=MODEL)
output_parser = StrOutputParser()

chain = prompt | model | output_parser
chain.invoke({"topic": "bears"})

In [22]:
chain.batch([{"topic": "bears"}, {"topic": "frogs"}])

for t in chain.stream({"topic": "bears"}):
    print(t)



Why
 do
 bears
 have
 hairy
 coats
?


Because
 they
 don
't
 believe
 in
 shaving
!



In [23]:
response = await chain.ainvoke({"topic": "bears"})
response

"Why don't bears like fast food?\n\nBecause they can never catch the burgers!"